# Goal

Четвёртый шаг в 40-ой линейке. Растим чемпиона с учётом полученного в рамках `35.4[cd]` знания, что на третьем шаге должен родится носитель навыка "умею проходить 5-ый уровень с nine_lives". Отключаем детерминизм, отключаем AMP на роллауте, чтобы было похоже на 35-ую линейку. Берём носитилей навыка из `40.3d`, треним с **`is_obs_usage_enforced=False`**. Диета для роллаута такая же, как и у `35.4[cd]`. Entropy loss тоже берём из `35.4[cd]`.

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_09.ipynb'

# OPTUNA_DECISIVE_METRIC

In [2]:
OPTUNA_DECISIVE_METRIC = 'test/levels_passed_mean'

# GRID_SEARCH_SPACE

In [3]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [4]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = False # нафиг детерминизм
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True  # без этого памяти будет жраться х1,5 и дольше, поэтому must-have
    HP.general.is_torch_amp_on_rollout = False  # AMP на роллауте тоже нафиг
    
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    wm_parent = '18d_world_model_09:183'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False

    parent = optuna_trial.suggest_categorical('parent', [
        '18n_ppo_tr_frostbite_09:286',
        '18n_ppo_tr_frostbite_09:289',
    ])
    HP.encoder.parent = dict(model=wm_parent, weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = parent
    HP.agent.sequence_length = 33 # задел на будущее!
    HP.agent.sequence_ages = [0, 1, 2, 3] # как в 40.1
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.break_on_level_passed = False
    HP.test.envs_count = 30
    HP.test.env_stories = None # just standard game

    # Rollout params
    HP.rollout.steps_count = 512 # how many steps to run in a single policy rolllout
    HP.rollout.env_stories = {
        # Main block
        '0:32':   (None, 'nine_lives'), 
    }
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.is_obs_usage_enforced = False # МИМИКРИРУЕМ ПОД frostbite_07, НАМЕРЕННО ОТЛИЧАЕМСЯ ОТ 40.4d

    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.03)') # ДЕЛАЕМ КАК В 35.4d
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization

    # Reporting
    HP.report.rollout_env_groups = None
    HP.report.follow_up_freq = 10
    return HP

# Results

Чисто визуально два запуска выглядит похоже. Ну может `40.4d` чуть повыше.

|что|`18n_study_40.4e`|`18n_study_40.4d`|
|---|---|---|
|levels_passed|<img src="./img/levels_passed_40.4e.png">|<img src="./img/levels_passed_40.4d.png">|
|episode_lp|<img src="./img/episode_lp_40.4e.png">|<img src="./img/episode_lp_40.4d.png">|
|episode_r|<img src="./img/episode_r_40.4e.png">|<img src="./img/episode_r_40.4d.png">|

**Выводы**
1) на 4-ом шаге `is_obs_usage_enforced=True` уже не актуален
2) что-то надо сделать, чтобы уйти от зависимости `is_obs_usage_enforced`

## Статистическая выжимка: `40.4e` vs `40.4d`

Чистый one-factor A/B: `set_hyperparameters` отличается **ровно одной строкой** — `HP.ppo.is_obs_usage_enforced` (`True` у `40.4d`, `False` у `40.4e`). Всё прочее построчно идентично: родители `09:286`/`09:289` (ось optuna, те же две руки в обоих), `lr const(2.5e-4)`, `ent const(0.03)`, `pred 0.1`, `consistency MSE 0.1`, диета `nine_lives`, 3M шагов, детерминизм OFF, AMP на роллауте OFF, `seq_len=33` / ages `[0,1,2,3]`, VH `09:183` заморожен. n=16/руку, все триалы complete.

**Итог: статистической разницы нет ни по решающей метрике, ни по каналу навыка.** Визуальное «`40.4d` чуть повыше» — на уровне шума (по среднему даже наоборот: `40.4e` выше). Единственный значимый факт — один схлопнувшийся ран (`:308`) именно в руке `obs_enforced=True`, что делает её дисперсию вшестеро больше; на средний уровень это не переносится.

### a) `test/levels_passed_mean` (решающая)

| метрика | `18n_study_40.4d` (obs_enforced=**True**) | `18n_study_40.4e` (obs_enforced=**False**) |
|---|---|---|
| Sample Size (n) | 16 | 16 |
| Mean Score | 2.9875 | 3.2354 |
| Best Value (model-trial) | 4.0333 (`:319`) | 3.6667 (`:331`) |
| Standard Deviation | 0.8828 | 0.3581 |
| Min | 0.0000 (`:308`, схлоп) | 2.0667 |
| Shapiro-Wilk (p) | 0.000 | 0.000 |
| Welch's t-test (p) | 0.311 | — |
| Mann-Whitney U (p) | 0.235 | — |
| Cohen's d | −0.368 (малый) | — |
| **Вердикт** | **н/з** (p ≥ 0.05) | — |

Без схлопнувшегося `:308`: Welch **p=0.722**, MWU **p=0.352** (средние 3.1867 vs 3.2354) — тем более н/з. MDE (независимые, n=16/16, 80% мощности, α=0.05): **0.690** при наблюдённой Δ=0.248 (без `:308` — MDE 0.392 при Δ=0.049) ⇒ тест не увидел бы и среднего эффекта: это «не отвергаем ноль», а не «разницы точно нет».

### b) Канал навыка 5-го уровня и потолок

| метрика | `40.4d` | `40.4e` | Fisher (p) |
|---|---|---|---|
| Носители: late-window `rollout/episode_lp_max ≥ 5` | 11/16 | 8/16 | 0.47 |
| `test/levels_passed_max ≥ 5` | 5/16 | 5/16 | 1.00 |
| Потолок `rollout/episode_lp_max` (max) | 7 | 7 | — |
| Потолок `test/levels_passed_max` (max) | 6 (`:313`) | 5 | — |

Оба канала — **н/з**. Пролом потолка (кап мутированной эры = 4) держится в обеих руках: `test/levels_passed_max ≥ 5` у 5/16 в каждой.

### c) Дисперсия — единственный значимый сигнал

| метрика | значение |
|---|---|
| var(`40.4d`) / var(`40.4e`) | 0.779 / 0.128 = **6.08×** |
| Bartlett (p) | **0.0012** |
| F-ratio (p) | **0.0006** |
| Levene (p, устойчив к выбросам) | 0.167 |
| без `:308`: отношение / Levene (p) | 1.21 / 0.30 |

Разницу держит один ран. `:308` — реальный коллапс: тест-серия доехала до конца (`test/levels_passed_mean` старт 2.767 → **0.0**; `test/levels_passed_max` 4 → **0.0**; `rollout/episode_r` max 13684 → last **150**). Это не шум-выброс, а разрушение политики внутри руки `obs_enforced=True`.

### d) Протуберанец-скрин (late-mean `rollout/episode_r` vs медиана когорты)

| студия | медиана | топ ratio |
|---|---|---|
| `40.4d` | 12303 | `:319` 1.22, `:314` 1.20 (ровно, отделения нет) |
| `40.4e` | 9324 | `:327` **1.58**, `:333` 1.53, `:325` 1.43, `:329` 1.40 |

Но MWU по late `episode_r` между студиями — **p=0.559**: уровни не отличаются, разница в **форме** (кластер у `40.4e` против ровного у `40.4d`), не в среднем. Плюс gap-over-runner-up у `:327` = 1.03 ⇒ по определению это кластер, а не одиночный протуберанец.

**Выводы**
1) **Статистической разницы между `40.4d` и `40.4e` нет**: решающая н/з (Welch p=0.31, MWU p=0.24; без `:308` — p=0.72/0.35), носители н/з (11/16 vs 8/16, p=0.47), `test_lpmax ≥ 5` н/з (5/16 vs 5/16, p=1.00). Т.е. на 4-м шаге `is_obs_usage_enforced` уже неактуален — качественный вывод подтверждается статистикой.
2) **Мощность низкая**: MDE 0.69 при Δ≈0.25 ⇒ «не отвергаем ноль», для уверенного закрытия нужен n≥40/руку.
3) **Единственный значимый факт — дисперсия**: `obs_enforced=True` дал один коллапс (`:308`), var 6× (Bartlett p=0.001), но Levene (p=0.17) держится на одном ране. Честная формулировка: *уровень не отличается; `obs_enforced=True`, возможно, добавляет хвостовой риск коллапса* — нужна репликация.
4) «Чуть повыше» `40.4d` — артефакт одного верхнего рана (`:319`=4.033) на фоне коллапса; по среднему `40.4e` выше (3.235 vs 2.988).


# System

In [5]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [6]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_09.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_09',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_40.4e/18n_study_40.4e.ipynb',
 'optuna_study_name': '18n_study_40.4e',
 'optuna_study_serial': '40.4e',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_40.4e/18n_study_40.4e.optuna'}

In [7]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [8]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [9]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [10]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [11]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 16
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-21 12:49:12,795] A new study created in Journal with name: 18n_study_40.4e


2026.10.21-12:49:13.173646     0.358 >> Model instance registered, version=320
2026.10.21-12:49:13.188791     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_09-launch320.ipynb"
2026.10.21-12:49:13.189026     0.005 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_09:320"; running launches=1
2026.10.21-12:49:44.230047     0.362 >> Model instance registered, version=321
2026.10.21-12:49:44.244887     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_09-launch321.ipynb"
2026.10.21-12:49:44.245679     0.001 >> 6.7 idle runners exist, submitted launch "18n_ppo_tr_frostbite_09:321"; running launches=2
2026.10.21-12:50:15.607040     0.348 >> Model instance registered, version=322
2026.10.21-12:50:15.621306     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_09-launch322.ipynb"
2026.10.21-12:50:15.622144     0.006 >> 6.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_09

In [12]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-21 14:30:26,987] Using an existing study with name '18n_study_40.4e' instead of creating a new one.


2026.10.21-14:30:26.989355     5.037 >> Study statistics: 
2026.10.21-14:30:26.991794     0.002 >> 	Number of finished trials: 16
2026.10.21-14:30:26.992455     0.001 >> 	Number of pruned trials: 0
2026.10.21-14:30:26.993730     0.001 >> 	Number of complete trials: 16
2026.10.21-14:30:26.994882     0.001 >> Best trial:
2026.10.21-14:30:26.995863     0.001 >> 	Value: 3.6666666666666665
2026.10.21-14:30:26.996209     0.000 >> 	Model version: 331
2026.10.21-14:30:26.996473     0.000 >> 	Params: 
2026.10.21-14:30:26.996712     0.000 >> 		parent: 18n_ppo_tr_frostbite_09:286


In [13]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.21-14:30:27.001759     0.005 >> Top trials:
2026.10.21-14:30:27.002295     0.001 >> 1) 18n_ppo_tr_frostbite_09:331, value=3.6666666666666665
2026.10.21-14:30:27.002650     0.000 >> 2) 18n_ppo_tr_frostbite_09:328, value=3.533333333333333
2026.10.21-14:30:27.002991     0.000 >> 3) 18n_ppo_tr_frostbite_09:320, value=3.433333333333333


In [14]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(3.2354166666666666), np.float64(0.3467044751017147))